<a href="https://colab.research.google.com/github/programminghistorian/ph-submissions/blob/gh-pages/assets/scraping-media-archived-web-wayback-machine/scraping-media-archived-web-wayback-machine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Build a tenacity decorator first. 
import tenacity

def give_up_on_failure(retry_state):
    print(f"Giving up after {retry_state.attempt_number} attempts.")
    return "skip"  # Do not retry anymore

retry = tenacity.retry(
    stop=tenacity.stop_after_attempt(10), 
    wait=tenacity.wait_exponential(multiplier=1, min=2, max=32),
    retry_error_callback=give_up_on_failure
)


In [ ]:
import time
import requests
from urllib.parse import urlencode

@retry
def download_cdx_data(url, sleep=1.5, from_timestamp=None, to_timestamp=None, filters=None, collapse=None):
    
    params = [('url', url)]
    if from_timestamp:
        params.append(('from', from_timestamp))
    if to_timestamp:
        params.append(('to', to_timestamp))
    if collapse:
        params.append(('collapse', collapse))
    if filters:
        for filter_item in filters:
            params.append(('filter', filter_item))

    cdx_url = f"https://web.archive.org/cdx/search/cdx?{urlencode(params)}"
    print(f"CDX URL: {cdx_url}")

    print(f"Fetching CDX data for: {url}")
    response = requests.get(cdx_url)
    response.raise_for_status()
    time.sleep(sleep)
    if response.status_code == 200:
        return response.text
    else:
        raise Exception(f"Failed to fetch CDX data for {url}: {response.status_code}")

In [ ]:
import csv
from pathlib import Path

csv_file = "nikkeibp-may2000-abridged.csv"

urls_data = []

with open(csv_file, mode='r', encoding='utf-8') as file:
    reader = csv.DictReader(file)
    urls_data = list(reader)

for url in urls_data: 
    # If the CDX data for the URL has already been downloaded, skip it.
    cdx_file_path = Path(f"data/urls/{url['url']}/cdx.csv")
    if cdx_file_path.exists():
        print(f"CDX data for {url['url']} already exists at {cdx_file_path}. Skipping download.")
        continue
    try:
        cdx_data = download_cdx_data(url['url'], from_timestamp="20000501000000", to_timestamp="20000531235959", filters=["statuscode:200"], collapse="digest")
        cdx_file_path.parent.mkdir(parents=True, exist_ok=True)
        with open(cdx_file_path, 'w', encoding='utf-8') as cdx_file:
            cdx_file.write(cdx_data)
        print(f"CDX data saved for {url['url']} at {cdx_file_path}")
    except Exception as e:
        print(f"Error fetching CDX data for {url['url']}: {e}")

In [ ]:
@retry
def download_archived_snapshot(url, timestamp, request_flag="id_", sleep=0.5):
    snapshot_url = f"https://web.archive.org/web/{timestamp}{request_flag}/{url}"
    print(f"Fetching archived snapshot for: {snapshot_url}")
    response = requests.get(snapshot_url, allow_redirects=True, stream=True)
    time.sleep(sleep)

    if response.status_code == 404 or response.status_code == 403: 
        return (response.status_code, None, response.headers, response.url)
    response.raise_for_status()

    # detect type of content
    content_type = response.headers.get("Content-Type")
    if (not content_type.startswith("text")):
        return (response.status_code, response.content, response.headers, response.url)
    
    # if the content is text, set the encoding to apparent encoding to ensure that the text is decoded correctly
    response.encoding = response.apparent_encoding
    return (response.status_code, response.text, response.headers, response.url)

In [ ]:
cdx_files = Path("data/urls/").glob("*/cdx.csv")
for cdx_file in cdx_files:
    print(f"Processing CDX file: {cdx_file}")
    snapshots = []
    with open(cdx_file, mode='r', encoding='utf-8') as file:
        reader = csv.reader(file, delimiter=" ")
        reader = list(reader)
        seen_digests = set()
        for row in reader:
            digest = row[5]
            if digest not in seen_digests:
                seen_digests.add(digest)
                snapshots.append(row)
    print(f"Found {len(snapshots)} unique snapshots in {cdx_file}")

    # Check if there are any snapshots already downloaded for this CDX file
    html_files_downloaded = list(cdx_file.parent.glob("*.html"))
    if html_files_downloaded:
        html_file_names = [file.stem for file in html_files_downloaded]
        snapshots = [snapshot for snapshot in snapshots if snapshot[1] not in html_file_names]
    print(f"Remaining snapshots to download: {len(snapshots)}")

    # Iterate through the snapshots
    for snapshot in snapshots:
        url, timestamp = snapshot[2], snapshot[1]
        print(f"Downloading snapshot for URL: {url} at timestamp: {timestamp}...")
        
        html_content = download_archived_snapshot(url, timestamp)
        if html_content == "skip":
            print(f"Skipping download for {url} at {timestamp} due to previous failure.")
            continue
        if html_content[0] != 200: # This is unlikely to happen, but if it does, we will raise an exception
            raise ValueError(f"Failed to download snapshot for {url} at {timestamp}: {html_content[0]}")
        # Save the HTML content to a file named after the timestamp of the snapshot
        html_file_path = cdx_file.parent / f"{timestamp}.html"
        
        with open(html_file_path, 'w', encoding='utf-8') as html_file:
            html_file.write(html_content[1])
        print(f"Snapshot saved at {html_file_path}")


In [ ]:
# load banner ad dimensions from banner-ad-dimensions.csv
with open('banner-ad-dimensions.csv', 'r') as file:
    reader = csv.DictReader(file)
    banner_dimensions = list(reader)  # Convert to list to read all rows

from urllib.parse import urlparse, urljoin
from bs4 import BeautifulSoup

def ensure_absolute_url(src, base_url):
    # Process the source URL to ensure it is absolute.
    # If the src is relative, it will be made absolute using the base URL.
    base_url = f'http://{base_url}' if not base_url.startswith(('http://')) else base_url
    if not (urlparse(src).netloc):
        # If src is relative, make it absolute using the base URL
        src = urljoin(base_url, src)
    elif src.startswith('//'):
        # If src is protocol-relative, add the HTTP scheme
        src = f"http:{src}"
    return src


def extract_ad_tags(html_content, html_base_url, banner_dimensions):

    soup = BeautifulSoup(html_content, 'html.parser')
    media_tags = soup.find_all('img') + soup.find_all('embed')
    
    extracted_ads = []

    for media in media_tags:
        width = media.get('width')
        height = media.get('height')

        if width and height:
            width = int(width)
            height = int(height)
            for banner in banner_dimensions:
                if (width == int(banner['width']) and height == int(banner['height'])):
                    # Process the src to ensure it is absolute
                    src = media.get('src')
                    src = ensure_absolute_url(src, html_base_url)
                    # if media is an <img> tag, get the ad link
                    # If media is an <object> or <embed> tag, return None
                    ad_href = None
                    if media.name == 'img':
                        ad_href = media.parent.get('href') if media.parent.name == 'a' else None
                    if ad_href:
                        ad_href = ensure_absolute_url(ad_href, html_base_url)
                    # Append the extracted banner ad details to the list
                    extracted_ads.append({
                        'tag': media.name,
                        'src': src,
                        'width': width,
                        'height': height,
                        'ad_href': ad_href,
                        'alt': media.get('alt', ''),
                    })
                    break  # Stop checking once a match is found

    return extracted_ads


In [ ]:
# Walk through the data directory and process each HTML file
data_dir = Path("data")

all_extracted_ads = []

for html_file in data_dir.glob("*/**/*.html"):
    timestamp = html_file.stem
    base_url = html_file.parent.name 
    print(f"Processing HTML file: {html_file} with timestamp: {timestamp}")
    # read the HTML content
    with open(html_file, 'r', encoding='utf-8') as file:
        html_content = file.read()

    extracted_ads = extract_ad_tags(html_content, base_url, banner_dimensions)
    for ad in extracted_ads:
        ad['web_page_snapshot_timestamp'] = timestamp
        ad['web_page_original_url'] = base_url
    all_extracted_ads.extend(extracted_ads)

# organize the extracted ads dictionary by unique URLs
from collections import defaultdict
organized_ads = defaultdict(list)
for ad in all_extracted_ads:
    organized_ads[ad['src']].append(ad)
# print out the number of unique URLs
print(f"Number of unique ad URLs: {len(organized_ads)}")

import hashlib
def md5_hash(url):
    """Return the MD5 hash of the given URL."""
    return hashlib.md5(url.encode('utf-8')).hexdigest()

# Now, we can create the final organized structure
final_organized_ads = {}
for url, appearances in organized_ads.items():
    final_organized_ads[md5_hash(url)] = {
        "src": url,
        "appearances": appearances
    }

# remove the `src` key from each appearance, as it is redundant
for entry in final_organized_ads.values():
    for appearance in entry['appearances']:
        if 'src' in appearance:
            del appearance['src']

# Print out some statistics about the tags used to display the ads
tag_counts = defaultdict(int)
for ad in all_extracted_ads:
    tag_counts[ad['tag']] += 1
print("Tag counts:")
for tag, count in tag_counts.items():
    print(f"{tag}: {count}")

# Save the final organized ads to a JSON file
import json
organized_extracted_ads = "data/organized_extracted_ads.json"
with open(organized_extracted_ads, 'w', encoding='utf-8') as jsonfile:
    json.dump(final_organized_ads, jsonfile, ensure_ascii=False, indent=4)
print(f"Ads data saved to {organized_extracted_ads}")


In [ ]:
import re
from io import BytesIO
from PIL import Image, UnidentifiedImageError

# Load image metadata
with open("data/organized_extracted_ads.json", 'r', encoding='utf-8') as f:
    final_organized_ads = json.load(f)

def get_image_file_extension_and_dimension(image_bytes_io):
    try:
        with Image.open(image_bytes_io) as img:
            return f".{img.format.lower()}", *img.size
    except (UnidentifiedImageError, Exception):
        return '.unk', None, None

def extract_timestamp_from_url(url):
    match = re.search(r'/web/(\d{14})', url)
    return match.group(1) if match else None

def create_placeholder(path, content=""):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with open(path, 'w') as f:
        f.write(content)

def skip_unavailable(ad_src_md5, timestamps, status_code):
    print(f"Skipping ad {ad_src_md5} due to status {status_code}.")
    for ts in timestamps:
        create_placeholder(f"data/ads/{ad_src_md5}/{ts}.{status_code}")

def save_ad(file_path, payload):
    Path(file_path).parent.mkdir(parents=True, exist_ok=True)
    with open(file_path, 'wb') as f:
        f.write(payload)

def download_one_ad_src(ad_data, ad_src_md5):
    src = ad_data['src']
    timestamps = [a['web_page_snapshot_timestamp'] for a in ad_data['appearances']]

    for appearance in ad_data['appearances']:
        ts = appearance['web_page_snapshot_timestamp']
        expected_w, expected_h = appearance.get('width'), appearance.get('height')
        if list(Path(f"data/ads/{ad_src_md5}").glob(f"{ts}*")):
            print(f"Already downloaded: {ad_src_md5} @ {ts}")
            continue

        print(f"Downloading {src} @ {ts}...")
        status, payload, headers, final_url = download_archived_snapshot(src, ts, "im_")
        print(f"Status: {status}, final URL: {final_url}")

        if status in [403, 404]:
            skip_unavailable(ad_src_md5, timestamps, status)
            break

        final_ts = extract_timestamp_from_url(final_url)
        if not isinstance(payload, bytes):
            create_placeholder(f"data/ads/{ad_src_md5}/{ts}-{final_ts}.not_bytes", str(payload))
            continue

        ext, actual_w, actual_h = get_image_file_extension_and_dimension(BytesIO(payload))
        if ext not in ['.jpg', '.jpeg', '.png', '.gif', '.bmp']:
            create_placeholder(f"data/ads/{ad_src_md5}/{ts}-{final_ts}.unsupported_file")
            continue
        
        if (actual_w, actual_h) != (expected_w, expected_h):
            print(f"Dimension mismatch: expected {expected_w}x{expected_h}, got {actual_w}x{actual_h}")
            save_ad(f"data/ads/{ad_src_md5}/{ts}-{final_ts}.{actual_w}x{actual_h}{ext}", payload)
        else:
            print(f"Saving {ad_src_md5} @ {ts}-{final_ts}{ext}")
            save_ad(f"data/ads/{ad_src_md5}/{ts}-{final_ts}{ext}", payload)

# Process all ads
for ad_src_md5, ad_data in final_organized_ads.items():
    download_one_ad_src(ad_data, ad_src_md5)


In [ ]:

# read the JSON file organized_extracted_ads.json again
with open('data/organized_extracted_ads.json', 'r', encoding='utf-8') as jsonfile:
    final_organized_ads = json.load(jsonfile)

# loop through final_organized_ads and compute the MD5 hash of each ad file
for ad_src_md5, ad_data in final_organized_ads.items():
    appearances = ad_data['appearances']
    for appearance in appearances:
        web_page_snapshot_timestamp = appearance['web_page_snapshot_timestamp']
        # get the file name of the ad
        file_name = f"data/ads/{ad_src_md5}/{web_page_snapshot_timestamp}*"
        # Find the download file
        ad_files = list(Path(file_name).parent.glob(Path(file_name).name))
        ad_file = ad_files[0]

        # get the extension of the ad file
        ext = ad_file.suffix[1:]  # remove the leading dot
        appearance['ad_snapshot_timestamp'] = ad_file.name.split('-')[1][:14] if '-' in ad_file.name else None
        if ext in ['jpg', 'jpeg', 'png', 'gif', 'bmp'] and ad_file.name.count('.') == 1:
            appearance['ad_snapshot_status'] = "scraped"
            appearance['ad_snapshot_path'] = str(ad_file)
        else:
            # if the filename contains two dots, it means the file has mismatched dimensions
            if ad_file.name.count('.') == 2:
                appearance['ad_snapshot_status'] = "mismatched_dimensions"
                appearance['ad_snapshot_path'] = str(ad_file)
            else:
                appearance['ad_snapshot_status'] = ad_file.suffix[1:]  # use the extension as the status for 404 and 403 errors

with open('data/results.json', 'w', encoding='utf-8') as jsonfile:
    json.dump(final_organized_ads, jsonfile, ensure_ascii=False, indent=4)

In [ ]:
# Of the 91 unique ads, how many were successfully scraped?
success_count = sum(1 for ad in final_organized_ads.values() if any(a['ad_snapshot_status'] == 'scraped' for a in ad['appearances']))
print(f"Number of successfully scraped ads: {success_count}")

In [ ]:
# For the 47 ads that were successfully scraped, calculate the average time skew in days
from datetime import datetime
success_ads = [ad for ad in final_organized_ads.values() if any(a['ad_snapshot_status'] == 'scraped' for a in ad['appearances'])]
time_skews = []
for ad in success_ads:
    for appearance in ad['appearances']:
        if appearance['ad_snapshot_status'] == 'scraped':
            web_page_snapshot_timestamp = appearance['web_page_snapshot_timestamp']
            ad_snapshot_timestamp = appearance['ad_snapshot_timestamp']
            if web_page_snapshot_timestamp and ad_snapshot_timestamp:
                web_page_ts = datetime.strptime(web_page_snapshot_timestamp, '%Y%m%d%H%M%S')
                ad_ts = datetime.strptime(ad_snapshot_timestamp, '%Y%m%d%H%M%S')
                time_skews.append((ad_ts - web_page_ts).days)
average_time_skew = sum(time_skews) / len(time_skews) if time_skews else 0
print(f"Average time skew for successfully scraped ads: {average_time_skew} days")

median_time_skew = sorted(time_skews)[len(time_skews) // 2] if time_skews else 0
print(f"Median time skew for successfully scraped ads: {median_time_skew} days")

In [ ]:
# Of the 91 unique ads, what is the distribution of their width x height?
width_x_height_distribution = defaultdict(int)
for ad in final_organized_ads.values():
    appearances = ad['appearances']
    width = appearances[0].get('width')
    height = appearances[0].get('height')
    if width and height:
        size = f"{width}x{height}"
        width_x_height_distribution[size] += 1

print("Width x Height distribution:")
for size, count in width_x_height_distribution.items():
    print(f"{size}: {count}")